In [ ]:
# =============================================================================
# 1. Importación de Librerías & Cliente
# =============================================================================
import pandas as pd
from datetime import datetime, timedelta
import pytz
from google.cloud import bigquery
from google.cloud import storage
from google.api_core.exceptions import NotFound
import os
clientBQ = bigquery.Client()
storage_client = storage.Client()

In [ ]:
# =============================================================================
# 2. Configuración de Fechas D-1 & Rutas
# =============================================================================

Zona = pytz.timezone('America/Lima')
peru_time = datetime.now(Zona)
peru_time_ayer = peru_time - timedelta(days=1)
var_fecha_ini = peru_time_ayer.strftime('%Y-%m-%d')
var_fecha_fin = peru_time.strftime('%Y-%m-%d')
fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

print(f"--- Fecha de inicio: {var_fecha_ini} ---")
print(f"--- Fecha de Fin: {var_fecha_fin} ---")
print(f"--- Fecha del proceso: {fecha_fin_dt} ---")


--- Fecha de inicio: 2026-05-04 ---
--- Fecha de Fin: 2026-05-05 ---
--- Fecha del proceso: 2026-05-05 00:00:00 ---


In [ ]:
## Variables de fecha como DataEntry
##var_fecha_ini = '2026-03-01'
##var_fecha_fin = '2026-03-31'
##fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

print(f"--- Fecha de inicio 1: {var_fecha_ini} ---")
print(f"--- Fecha de Fin 2: {var_fecha_fin} ---")
print(f"--- Fecha del proceso 3: {fecha_fin_dt} ---")

--- Fecha de inicio 1: 2026-05-04 ---
--- Fecha de Fin 2: 2026-05-05 ---
--- Fecha del proceso 3: 2026-05-05 00:00:00 ---


In [ ]:
# =============================================================================
# 2. Configuración de Variables de entorno
# =============================================================================
var_anho = fecha_fin_dt.strftime('%Y')
var_mes = fecha_fin_dt.strftime('%m')
var_fecha_file = fecha_fin_dt.strftime('%Y%m%d')

print(f"--- Año del proceso {var_anho} ---")
print(f"--- Mes del proceso: {var_mes} ---")
print(f"--- Fecha del archivo: {var_fecha_file} ---")

--- Año del proceso 2026 ---
--- Mes del proceso: 05 ---
--- Fecha del archivo: 20260505 ---


In [ ]:
# =============================================================================
# 3. Configuración de Rutas y Parámetros
# =============================================================================
bucket_name = "adls-reportes"
ruta_base = f"Data/Marca/Comunicacion_anti_fraude/{var_anho}/{var_mes}/"
nombre_final = f"comunicacion_anti_fraude_{var_fecha_file}.csv"  # <-- CSV
proyecto_storage = "prd-izipay-data-storage-pv"
proyecto_sensitivo = "prd-izipay-data-sensitive"

In [ ]:
# =============================================================================
# 4. Creación de Tabla Temporal en BigQuery
# =============================================================================

temp_table_id = f"prd-izipay-data-operation.master_stage_financial.temp_abono_detalle_{var_fecha_file}"

query_temp = f"""
CREATE OR REPLACE TABLE `{temp_table_id}` AS
with ultimo_periodo_segmentacion as (
    select codigo, segmento
    from `{proyecto_storage}.raw_dataentry_planeamiento.segmentacion`
    where periodo = (
        select max(periodo)
        from `{proyecto_storage}.raw_dataentry_planeamiento.segmentacion`
    )
)
select distinct
    trim(c.document_number) as RUC,
    trim(AEAD.DECRYPT_STRING(e.key, a.correo_representante_legal, e.constant)) as correo_representante_legal,
    trim(AEAD.DECRYPT_STRING(t.key, a.telefono_comercio, t.constant)) as telefono,
    seg.segmento,
    a.cod_situacion_comercio as situacion,
    a.flag_lpdp as flag_lpdp
from `{proyecto_storage}.master_party.m_comercio` a
left join `{proyecto_sensitivo}.master_pii.iden_party_data_control` c on (a.party_id_izi = c.party_id_izi)
left join `{proyecto_sensitivo}.secure_secrets.config_protected_data` e on (1=1 and e.code = 'C_EMAIL')
left join `{proyecto_sensitivo}.secure_secrets.config_protected_data` ec on (1=1 and ec.code = 'C_FULL_NAME')
left join `{proyecto_sensitivo}.secure_secrets.config_protected_data` t on (1=1 and t.code = 'C_TELEPHONE')
left join ultimo_periodo_segmentacion seg on (a.cod_comercio = seg.codigo)
where a.cod_situacion_comercio not in ('3', '9')
    and a.cod_situacion_comercio is not null
    and a.compania in ('PMP','IZIPAY')
    and a.nom_producto not in ('CAJERO CORRESPONSAL','INTEROPERABILIDAD VISANET','VENDEMAS','IZIPAY YA')
    and a.flag_parque = true
"""

print(f"Creando tabla temporal: {temp_table_id} ...")
clientBQ.query(query_temp).result()
print("✅ Tabla temporal creada correctamente.")
print(f"Query Ejecutado:  ...")
print(f"{query_temp} ")

Creando tabla temporal: prd-izipay-data-operation.master_stage_financial.temp_abono_detalle_20260505 ...
✅ Tabla temporal creada correctamente.
Query Ejecutado:  ...

CREATE OR REPLACE TABLE `prd-izipay-data-operation.master_stage_financial.temp_abono_detalle_20260505` AS
with ultimo_periodo_segmentacion as (
    select codigo, segmento
    from `prd-izipay-data-storage-pv.raw_dataentry_planeamiento.segmentacion`
    where periodo = (
        select max(periodo) 
        from `prd-izipay-data-storage-pv.raw_dataentry_planeamiento.segmentacion`
    )
)
select distinct
    c.document_number as RUC,
    -- AEAD.DECRYPT_STRING(e.key, a.correo_representante_legal, e.constant) as correo_representante_legal,
    -- AEAD.DECRYPT_STRING(t.key, a.telefono_comercio, t.constant) as telefono,
    seg.segmento,
    a.cod_situacion_comercio as situacion,
    a.flag_lpdp as flag_lpdp
from `prd-izipay-data-storage-pv.master_party.m_comercio` a
left join `dev-izipay-data-storage.master_pii.iden_party_da

In [ ]:
# =============================================================================
# 5. Exportación desde Tabla Temporal a GCS (CSV)
# =============================================================================

uri_temporal = f"gs://{bucket_name}/{ruta_base}temp_{var_fecha_file}_*.csv"  # <-- CSV
print(f"Exportando desde tabla temporal a: {uri_temporal} ...")

query_export = f"""
EXPORT DATA OPTIONS (
  uri = '{uri_temporal}',
  format = 'CSV',
  overwrite = true,
  header = true
) AS
SELECT * FROM `{temp_table_id}`;
"""

clientBQ.query(query_export).result()
print("✅ Exportación a GCS completada.")

# Eliminar tabla temporal
clientBQ.delete_table(temp_table_id)
print(f"🧹 Tabla temporal eliminada: {temp_table_id}")


Exportando desde tabla temporal a: gs://adls-reportes/Data/MARCA/Base_Comercio/2026/05/temp_20260505_*.csv ...
✅ Exportación a GCS completada.
🧹 Tabla temporal eliminada: prd-izipay-data-operation.master_stage_financial.temp_abono_detalle_20260505


In [ ]:
# =============================================================================
# 6. Consolidación incremental a CSV comprimido (sin cargar todo en memoria)
# =============================================================================
# Escribe part a part en un archivo local temporal y luego sube a GCS.
# Así evitamos acumular todo el dataset en RAM.

print("Consolidando archivos CSV en uno solo (modo incremental)...")
bucket = storage_client.bucket(bucket_name)
prefix_temp = f"{ruta_base}temp_{var_fecha_file}_"

blobs = list(bucket.list_blobs(prefix=prefix_temp))

if not blobs:
    print("⚠️ AVISO: No se encontraron archivos temporales para consolidar.")
else:
    local_tmp = f"/tmp/{nombre_final}"
    primera_parte = True

    for blob in blobs:
        uri_parte = f"gs://{bucket_name}/{blob.name}"
        df_part = pd.read_csv(uri_parte)
        df_part.to_csv(
            local_tmp,
            mode='a',
            index=False,
            header=primera_parte,
            sep=';'
        )
        primera_parte = False
        del df_part

    # Subir archivo consolidado a GCS
    ruta_final_full = f"{ruta_base}{nombre_final}"
    blob_final = bucket.blob(ruta_final_full)
    blob_final.upload_from_filename(local_tmp)
    os.remove(local_tmp)

    # Borrar parts temporales de GCS
    for blob in blobs:
        try:
            bucket.blob(blob.name).delete()
        except Exception as e:
            print(f"⚠️ No se pudo borrar {blob.name}: {e}")

    print(f"✅ ÉXITO: Archivo único creado en: gs://{bucket_name}/{ruta_final_full}")
    print(f"🧹 {len(blobs)} archivos temporales eliminados")

Consolidando archivos CSV en uno solo (modo incremental)...
✅ ÉXITO: Archivo único creado en: gs://adls-reportes/Data/MARCA/Base_Comercio/2026/05/comunicacion_anti_fraude_20260505.csv
🧹 1 archivos temporales eliminados
